# Benchmark Qwen ASR 1.7B: PiSSA và AdaLoRA trên Zalo train

Chạy **hai adapter full**, checkpoint tốt nhất theo `best_epoch.txt`, trên tất cả
mẫu có nhãn trong `train.jsonl`. So sánh WER/CER raw và normalized, coverage,
latency và RTF. Giữ audio gốc, cùng giải mã greedy Vietnamese, tối đa 768 token.

Đây là notebook **local**, cần CUDA cho inference. Môi trường Transformers 5 / PEFT
được tạo riêng, không cài đè vào môi trường Qwen SDK của baseline.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "src/benchmark_baseline_asr").is_dir()),
    Path("/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/Zalo-AI_Challenge"),
).resolve()
MODELS_ROOT = Path("/run/media/fernir/Data/Documents/HCMUS/Research/models")
DATASET_ROOT = PROJECT_ROOT / "data/train+public_test/train"
OUTPUT_ROOT = PROJECT_ROOT / "output/qwen-adapter-benchmark"
VENV_ROOT = OUTPUT_ROOT / "venv"
VENV_PYTHON = VENV_ROOT / "bin/python"
GPU_IDS = [0]  # [0, 1] nếu có hai GPU
RUN_ID = "qwen-full-train-v1"  # Giữ nguyên để resume; đổi khi đổi cấu hình/trọng số.
SMOKE_RUN_ID = RUN_ID + "-smoke"
ENV_FILE = PROJECT_ROOT / ".env"  # Chỉ đọc HF_TOKEN; không hiển thị nội dung.
INSTALL_DEPENDENCIES = True  # False khi môi trường riêng đã cài xong.
assert (PROJECT_ROOT / "src/benchmark_baseline_asr").is_dir(), PROJECT_ROOT
assert MODELS_ROOT.is_dir(), MODELS_ROOT
assert (DATASET_ROOT / "train.jsonl").is_file(), DATASET_ROOT
print("Project:", PROJECT_ROOT)


## 1. Môi trường riêng bằng pip

Cài đặt chỉ tác động virtualenv trong thư mục output. Kernel hiện tại chỉ điều phối
subprocess và hiển thị kết quả; inference chạy bằng Python của virtualenv.
Lần đầu cần Internet để cài thư viện và tải base model đúng revision.


In [ ]:
if not VENV_PYTHON.is_file():
    subprocess.run([sys.executable, "-m", "venv", str(VENV_ROOT)], check=True)
pip_check = subprocess.run(
    [str(VENV_PYTHON), "-m", "pip", "--version"], capture_output=True, text=True,
)
if pip_check.returncode:
    subprocess.run([str(VENV_PYTHON), "-m", "ensurepip", "--upgrade"], check=True)
if INSTALL_DEPENDENCIES:
    subprocess.run([
        str(VENV_PYTHON), "-m", "pip", "install", "-r",
        str(PROJECT_ROOT / "requirements/qwen-adapter-benchmark.txt"),
    ], check=True)

child_env = os.environ.copy()
child_env["PYTHONPATH"] = str(PROJECT_ROOT / "src")
child_env["PYTHONUNBUFFERED"] = "1"

def run_cli(run_id, *, prepare_only=False, limit=None):
    command = [
        str(VENV_PYTHON), "-m", "benchmark_baseline_asr.workflow.benchmark_qwen_adapters",
        "--models-root", str(MODELS_ROOT), "--dataset-root", str(DATASET_ROOT),
        "--output-root", str(OUTPUT_ROOT), "--run-id", run_id,
        "--gpu-ids", *map(str, GPU_IDS),
    ]
    if ENV_FILE.is_file():
        command.extend(["--env-file", str(ENV_FILE)])
    if prepare_only:
        command.append("--prepare-only")
    if limit is not None:
        command.extend(["--limit", str(limit)])
    subprocess.run(command, cwd=PROJECT_ROOT, env=child_env, check=True)
    return OUTPUT_ROOT / "runs" / run_id


## 2. Kiểm tra toàn bộ train và adapter (CPU)

Không tải trọng số base ở bước này. Kiểm tra checkpoint, revision, SHA-256,
manifest và WAV; mặc định không giới hạn số mẫu. Các file WAV ngoài manifest
không được tự thêm vào benchmark.


In [ ]:
RUN_DIR = run_cli(RUN_ID, prepare_only=True)
manifest = [json.loads(line) for line in (RUN_DIR / "manifest.jsonl").read_text().splitlines() if line]
identity = json.loads((RUN_DIR / "adapter_benchmark.json").read_text())
print("Samples:", len(manifest))
print("Audio hours:", round(sum(row["duration_seconds"] for row in manifest) / 3600, 3))
for model in identity["models"]:
    settings = model["settings"]
    print(model["key"], "epoch", settings["epoch"], settings["adapter_path"])
    print("Base:", settings["repo_id"], "revision:", settings["revision"])


## 3. Kiểm tra GPU và smoke test hai mẫu

Smoke test nạp cả hai adapter và chạy hết chuỗi inference → metrics → comparison.
Dùng run ID riêng để giữ kết quả toàn tập độc lập. Nếu bước này lỗi, sửa lỗi trước
khi chạy cell full benchmark.


In [ ]:
subprocess.run([
    str(VENV_PYTHON), "-c",
    "import torch, transformers, peft; "
    "print('Transformers:', transformers.__version__, 'PEFT:', peft.__version__); "
    "print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]); "
    "assert torch.cuda.is_available(), 'Cần CUDA để chạy benchmark inference'",
], cwd=PROJECT_ROOT, env=child_env, check=True)
SMOKE_DIR = run_cli(SMOKE_RUN_ID, limit=2)
smoke_results = json.loads((SMOKE_DIR / "comparison.json").read_text())
assert all(row["result_complete"] and row["successful_samples"] == 2 for row in smoke_results)
print("Smoke test completed for both adapters")


## 4. Full benchmark train

Cell này chạy toàn bộ manifest, không có `--limit`. Hai adapter dùng chung base
snapshot và chạy lần lượt; mỗi model chia dữ liệu trên các GPU đã chọn. Chạy lại
cell với cùng RUN_ID để tiếp tục mẫu chưa hoàn thành. Không đổi adapter, dữ liệu
hoặc cấu hình trong cùng lần chạy. Dừng kernel không đảm bảo đã dừng các worker;
kiểm tra tiến trình trước khi khởi chạy lần nữa.


In [ ]:
RUN_DIR = run_cli(RUN_ID)


## 5. So sánh kết quả

Ưu tiên normalized WER/CER thấp hơn và coverage = 100%. Đây là **corpus WER/CER**,
không phải trung bình WER/CER từng câu. Normalization giữ dấu tiếng Việt và cách
viết số; CER tính cả khoảng trắng. Không bật BERTScore hoặc DNSMOS ở experiment này.


In [ ]:
import pandas as pd
from IPython.display import display

comparison = pd.DataFrame(json.loads((RUN_DIR / "comparison.json").read_text()))
columns = ["model", "expected_samples", "successful_samples", "result_complete", "coverage",
           "raw_wer", "normalized_wer", "raw_cer", "normalized_cer",
           "latency_mean_seconds", "latency_p95_seconds", "rtf_sum_latency"]
display(comparison[[c for c in columns if c in comparison]].sort_values("normalized_wer"))
assert comparison["result_complete"].all(), "Kết quả chưa đủ mẫu; kiểm tra errors rồi resume."
print("CSV:", RUN_DIR / "comparison.csv")


## 6. Xem và nghe theo sample

Đổi SAMPLE_ID để đối chiếu lời tham chiếu với hai prediction. Audio được phát từ
file gốc; hai model không thay đổi waveform.


In [ ]:
from IPython.display import Audio

SAMPLE_ID = "train_0001"
sample = next(row for row in manifest if row["sample_id"] == SAMPLE_ID)
print("Reference:", sample["reference"])
display(Audio(filename=sample["audio_path"]))
for model in identity["models"]:
    path = RUN_DIR / model["key"] / "predictions.jsonl"
    predictions = [json.loads(line) for line in path.read_text().splitlines() if line]
    prediction = next((r for r in predictions if r["sample_id"] == SAMPLE_ID), None)
    print(model["key"], ":", prediction["text"] if prediction else "Chưa có prediction")


## 7. Kiểm tra lỗi còn lại

Lỗi của các lần chạy trước vẫn được giữ để truy vết. Chỉ liệt kê lỗi có sample
chưa xuất hiện trong predictions của model; sau khi sửa lỗi, chạy lại cell full.


In [ ]:
for model in identity["models"]:
    model_dir = RUN_DIR / model["key"]
    prediction_path = model_dir / "predictions.jsonl"
    successful = {json.loads(line)["sample_id"] for line in prediction_path.read_text().splitlines() if line} if prediction_path.is_file() else set()
    errors = []
    for path in model_dir.glob("worker_*/errors.jsonl"):
        errors.extend(json.loads(line) for line in path.read_text().splitlines() if line)
    unresolved = [row for row in errors if row["sample_id"] not in successful]
    print(model["key"], "unresolved error records:", len(unresolved))
    if unresolved:
        display(pd.DataFrame(unresolved).head(20))
